In [3]:
import pandas as pd
import numpy as np
import datetime as dt
from sklearn.linear_model import LinearRegression, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def regression_metrics(y_true, y_pred, shift, alphaval, ratio):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)

    # MAPE (avoid division by zero by adding a tiny number)
    eps = 1e-6
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))) * 100.0

    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE(%)": mape, "Minutes Shifted": shift*5, "Alpha Value": alphaval, "L1 Ratio": ratio}

def byHour(df):

  df["DateTime"] = pd.to_datetime(df["DateTime"])

  df['Hour'] = df['DateTime'].apply(lambda n: n.hour)
  df['Date'] = df['DateTime'].apply(lambda n: n.date())
  dates = df['Date'].unique()

  res = []
  for d in dates:
    temp = df.where(df['Date'] == d).groupby(['Hour'])[['Speed',
                                                        'Volume',
                                                        'Volume Per Lane',
                                                        'Occupancy']].mean().reset_index()
    temp['Date'] = d
    temp['DateTime'] = temp.apply(lambda row:
                                  pd.to_datetime(f"{row['Date']}" + " " +
                                  f"{dt.time(hour=int(row['Hour'])).strftime('%H:%M:%S')}"), axis = 1)
    res.append(temp)

  return res

def timelag(df1, df2, shiftn):
  df1["DateTime"] = pd.to_datetime(df1["DateTime"])
  df2["DateTime"] = pd.to_datetime(df2["DateTime"])
  df1[['SpeedLag1',
       'VolLag1',
       'VPLLag1',
       'OccLag1']] = df2[['Speed',
                          'Volume',
                          'Volume Per Lane',
                          'Occupancy']].shift(shiftn)
  df1 = df1.dropna().reset_index(drop=True)

  if 'Unnamed: 0' in df1.columns:
    df1 = df1.drop('Unnamed: 0', axis = 1)
  return df1

def dfSort(df):
  sorted = df.dropna().reset_index(drop=True)
  cutoff_train = sorted["DateTime"].quantile(0.70)
  cutoff_val= sorted["DateTime"].quantile(0.85)

  train = sorted[sorted["DateTime"] < cutoff_train]
  val   = sorted[(sorted["DateTime"] >= cutoff_train) & (sorted["DateTime"] < cutoff_val)]
  test  = sorted[sorted["DateTime"] >= cutoff_val]
  return sorted, train, val, test

def trainModel(traindat, valdat, testdat, model, pred_val, shiftn, alpha, ratio):
  xval = traindat[['SpeedLag1', 'VolLag1', 'VPLLag1', 'OccLag1']]
  model.fit(xval, traindat[pred_val].to_frame())

  y_val   = valdat[pred_val].values
  y_test = testdat[pred_val].values

  pred_val = model.predict(valdat[['SpeedLag1', 'VolLag1', 'VPLLag1', 'OccLag1']])
  pred_test = model.predict(testdat[['SpeedLag1', 'VolLag1', 'VPLLag1', 'OccLag1']])

  lin_val = regression_metrics(y_val, pred_val, shiftn, alpha, ratio)
  lin_test = regression_metrics(y_test, pred_test, shiftn, alpha, ratio)

  res = pd.DataFrame([lin_val, lin_test], index=[f"{model} (Val)", f"{model} (Test)"])
  return res


In [4]:
filename_i5 = '005es16732_loop_cloutput.csv'
filename_520 = '520es00972_loop_cloutput.csv'

dfi5 = pd.read_csv(filename_i5)
df520 = pd.read_csv(filename_520)

In [82]:
shiftns =  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
alphavals = 0.1 #[0.1, 1, 5, 10, 20, 100, 200, 500] 
l1ratios = 0.45 #[0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]

spdfulllr = []
volfulllr = []
spdfulllas = []
volfulllas = []
spdfulleln = []
volfulleln = []

#Testing of diffrent time shifts

for i in range(len(shiftns)):
    lr = LinearRegression()
    lrlasso = Lasso(alpha=alphavals)
    lreln = ElasticNet(alpha=alphavals, l1_ratio = l1ratios)
    dftemp = timelag(dfi5.copy(), df520.copy(), shiftns[i])
    dftemp_sorted, trainm, valm, testm = dfSort(dftemp)
    resvollr = trainModel(trainm, valm, testm, lr, 'Volume', shiftns[i], alphavals, l1ratios)
    resspdlr = trainModel(trainm, valm, testm, lr, 'Speed', shiftns[i], alphavals, l1ratios)
    resvollas = trainModel(trainm, valm, testm, lrlasso, 'Volume', shiftns[i], alphavals, l1ratios)
    resspdlas = trainModel(trainm, valm, testm, lrlasso, 'Speed', shiftns[i], alphavals, l1ratios)
    resvoleln = trainModel(trainm, valm, testm, lreln, 'Volume', shiftns[i], alphavals, l1ratios)
    resspdeln = trainModel(trainm, valm, testm, lreln, 'Speed', shiftns[i], alphavals, l1ratios)
    volfulllr.append(resvollr)
    spdfulllr.append(resspdlr)
    volfulllas.append(resvollas)
    spdfulllas.append(resspdlas)
    volfulleln.append(resvoleln)
    spdfulleln.append(resspdeln)

#Testing of different alpha values

# for i in range(len(alphavals)):
#     lr = LinearRegression()
#     lrlasso = Lasso(alpha=alphavals[i])
#     lreln = ElasticNet(alpha=alphavals[i], l1_ratio = l1ratios)
#     dftemp = timelag(dfi5.copy(), df520.copy(), shiftns)
#     dftemp_sorted, trainm, valm, testm = dfSort(dftemp)
#     resvollr = trainModel(trainm, valm, testm, lr, 'Volume', shiftns, alphavals[i], l1ratios)
#     resspdlr = trainModel(trainm, valm, testm, lr, 'Speed', shiftns, alphavals[i], l1ratios)
#     resvollas = trainModel(trainm, valm, testm, lrlasso, 'Volume', shiftns, alphavals[i], l1ratios)
#     resspdlas = trainModel(trainm, valm, testm, lrlasso, 'Speed', shiftns, alphavals[i], l1ratios)
#     resvoleln = trainModel(trainm, valm, testm, lreln, 'Volume', shiftns, alphavals[i], l1ratios)
#     resspdeln = trainModel(trainm, valm, testm, lreln, 'Speed', shiftns, alphavals[i], l1ratios)
#     volfulllr.append(resvollr)
#     spdfulllr.append(resspdlr)
#     volfulllas.append(resvollas)
#     spdfulllas.append(resspdlas)
#     volfulleln.append(resvoleln)
#     spdfulleln.append(resspdeln)

#Testing of different L1 ratios

# for i in range(len(l1ratios)):
#     lr = LinearRegression()
#     lrlasso = Lasso(alpha=alphavals)
#     lreln = ElasticNet(alpha=alphavals, l1_ratio = l1ratios[i])
#     dftemp = timelag(dfi5.copy(), df520.copy(), shiftns)
#     dftemp_sorted, trainm, valm, testm = dfSort(dftemp)
#     resvollr = trainModel(trainm, valm, testm, lr, 'Volume', shiftns, alphavals, l1ratios[i])
#     resspdlr = trainModel(trainm, valm, testm, lr, 'Speed', shiftns, alphavals, l1ratios[i])
#     resvollas = trainModel(trainm, valm, testm, lrlasso, 'Volume', shiftns, alphavals, l1ratios[i])
#     resspdlas = trainModel(trainm, valm, testm, lrlasso, 'Speed', shiftns, alphavals, l1ratios[i])
#     resvoleln = trainModel(trainm, valm, testm, lreln, 'Volume', shiftns, alphavals, l1ratios[i])
#     resspdeln = trainModel(trainm, valm, testm, lreln, 'Speed', shiftns, alphavals, l1ratios[i])
#     volfulllr.append(resvollr)
#     spdfulllr.append(resspdlr)
#     volfulllas.append(resvollas)
#     spdfulllas.append(resspdlas)
#     volfulleln.append(resvoleln)
#     spdfulleln.append(resspdeln)

spdfinlr = pd.concat(spdfulllr)
volfinlr = pd.concat (volfulllr)
spdfinlas = pd.concat(spdfulllas)
volfinlas = pd.concat (volfulllas)
spdfineln = pd.concat(spdfulleln)
volfineln = pd.concat (volfulleln)


In [83]:
print('Linear Regression \nTraining to Predict Speed:\n')
display(spdfinlr)

Linear Regression 
Training to Predict Speed:



,MAE,RMSE,R2,MAPE(%),Minutes Shifted,Alpha Value,L1 Ratio
LinearRegression() (Val),12.088561,14.162745,0.350764,59.528419,0,0.1,0.45
LinearRegression() (Test),12.881480,15.140483,0.295720,55.572277,0,0.1,0.45
LinearRegression() (Val),12.025565,14.112986,0.355182,59.562077,5,0.1,0.45
LinearRegression() (Test),12.801834,15.058821,0.303346,55.624608,5,0.1,0.45
LinearRegression() (Val),11.981412,14.060064,0.360009,59.595402,10,0.1,0.45
LinearRegression() (Test),12.724134,14.983948,0.310256,55.668749,10,0.1,0.45
LinearRegression() (Val),11.910869,13.992927,0.366106,59.634929,15,0.1,0.45
LinearRegression() (Test),12.652614,14.916633,0.316439,55.714951,15,0.1,0.45
LinearRegression() (Val),11.847119,13.944326,0.370431,59.661550,20,0.1,0.45
LinearRegression() (Test),12.586760,14.850795,0.322460,55.752384,20,0.1,0.45


In [86]:
print('Linear Regression \nTraining to Predict Volume:\n')
display(volfinlr)

Linear Regression 
Training to Predict Volume:



,MAE,RMSE,R2,MAPE(%),Minutes Shifted,Alpha Value,L1 Ratio
LinearRegression() (Val),13.282284,17.946011,0.718864,4.893931e+07,0,0.1,0.45
LinearRegression() (Test),12.851190,17.263422,0.727148,2.221600e+07,0,0.1,0.45
LinearRegression() (Val),12.993427,17.618926,0.728987,4.894967e+07,5,0.1,0.45
LinearRegression() (Test),12.569994,16.897052,0.738622,2.220518e+07,5,0.1,0.45
LinearRegression() (Val),12.781176,17.368122,0.736647,4.896349e+07,10,0.1,0.45
LinearRegression() (Test),12.314333,16.645158,0.746357,2.219457e+07,10,0.1,0.45
LinearRegression() (Val),12.630434,17.250904,0.740190,4.898241e+07,15,0.1,0.45
LinearRegression() (Test),12.164361,16.444818,0.752426,2.218558e+07,15,0.1,0.45
LinearRegression() (Val),12.479632,17.061102,0.745864,4.899995e+07,20,0.1,0.45
LinearRegression() (Test),12.030709,16.275904,0.757486,2.218370e+07,20,0.1,0.45


In [85]:
print('Linear Regression w/ Lasso Regularization\nTraining to Predict Speed:\n')
display(spdfinlas)

Linear Regression w/ Lasso Regularization
Training to Predict Speed:



,MAE,RMSE,R2,MAPE(%),Minutes Shifted,Alpha Value,L1 Ratio
Lasso(alpha=0.1) (Val),12.115476,14.181126,0.349077,43.949554,0,0.1,0.45
Lasso(alpha=0.1) (Test),12.923874,15.151470,0.294698,42.512000,0,0.1,0.45
Lasso(alpha=0.1) (Val),12.050543,14.132200,0.353425,43.710642,5,0.1,0.45
Lasso(alpha=0.1) (Test),12.839116,15.065494,0.302728,42.288070,5,0.1,0.45
Lasso(alpha=0.1) (Val),12.006563,14.079786,0.358212,43.500445,10,0.1,0.45
Lasso(alpha=0.1) (Test),12.758728,14.990280,0.309673,42.059609,10,0.1,0.45
Lasso(alpha=0.1) (Val),11.936682,14.014501,0.364150,43.283616,15,0.1,0.45
Lasso(alpha=0.1) (Test),12.682570,14.922408,0.315910,41.861499,15,0.1,0.45
Lasso(alpha=0.1) (Val),11.866701,13.964108,0.368644,43.060618,20,0.1,0.45
Lasso(alpha=0.1) (Test),12.615437,14.857877,0.321814,41.664210,20,0.1,0.45


In [88]:
print('Training to Predict Volume:\n')
display(volfinlas)

Training to Predict Volume:



,MAE,RMSE,R2,MAPE(%),Minutes Shifted,Alpha Value,L1 Ratio
Lasso(alpha=0.1) (Val),13.293473,17.950423,0.718726,4.534736e+07,0,0.1,0.45
Lasso(alpha=0.1) (Test),12.864478,17.266275,0.727058,3.750908e+06,0,0.1,0.45
Lasso(alpha=0.1) (Val),13.004435,17.624287,0.728822,4.601818e+07,5,0.1,0.45
Lasso(alpha=0.1) (Test),12.583446,16.899452,0.738548,4.623327e+06,5,0.1,0.45
Lasso(alpha=0.1) (Val),12.791241,17.373341,0.736489,4.596260e+07,10,0.1,0.45
Lasso(alpha=0.1) (Test),12.327082,16.645115,0.746358,4.577434e+06,10,0.1,0.45
Lasso(alpha=0.1) (Val),12.640826,17.255866,0.740041,4.623712e+07,15,0.1,0.45
Lasso(alpha=0.1) (Test),12.173455,16.445259,0.752413,5.028007e+06,15,0.1,0.45
Lasso(alpha=0.1) (Val),12.488053,17.065963,0.745720,4.682331e+07,20,0.1,0.45
Lasso(alpha=0.1) (Test),12.039486,16.275618,0.757494,5.390789e+06,20,0.1,0.45


In [87]:
print('ElasticNet Linear Regression\nTraining to Predict Speed:\n')
display(spdfineln)

ElasticNet Linear Regression
Training to Predict Speed:



,MAE,RMSE,R2,MAPE(%),Minutes Shifted,Alpha Value,L1 Ratio
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Val)",12.116080,14.181303,0.349061,43.950141,0,0.1,0.45
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Test)",12.925393,15.152730,0.294581,42.513938,0,0.1,0.45
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Val)",12.051244,14.132442,0.353403,43.711465,5,0.1,0.45
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Test)",12.840658,15.066716,0.302615,42.290028,5,0.1,0.45
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Val)",12.007221,14.079969,0.358195,43.501150,10,0.1,0.45
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Test)",12.760274,14.991512,0.309559,42.061610,10,0.1,0.45
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Val)",11.937431,14.014695,0.364133,43.284578,15,0.1,0.45
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Test)",12.684070,14.923627,0.315798,41.863417,15,0.1,0.45
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Val)",11.867321,13.964200,0.368636,43.061350,20,0.1,0.45
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Test)",12.616963,14.859058,0.321706,41.666232,20,0.1,0.45


In [89]:
print('Training to Predict Volume:\n')
display(volfineln)

Training to Predict Volume:



,MAE,RMSE,R2,MAPE(%),Minutes Shifted,Alpha Value,L1 Ratio
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Val)",13.293821,17.950693,0.718717,4.534902e+07,0,0.1,0.45
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Test)",12.865177,17.266635,0.727046,3.751449e+06,0,0.1,0.45
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Val)",13.004879,17.624568,0.728813,4.601959e+07,5,0.1,0.45
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Test)",12.584206,16.899866,0.738535,4.624525e+06,5,0.1,0.45
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Val)",12.791669,17.373573,0.736482,4.596402e+07,10,0.1,0.45
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Test)",12.327841,16.645533,0.746346,4.578091e+06,10,0.1,0.45
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Val)",12.641215,17.256137,0.740033,4.623908e+07,15,0.1,0.45
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Test)",12.174276,16.445684,0.752400,5.027987e+06,15,0.1,0.45
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Val)",12.488436,17.066163,0.745714,4.682512e+07,20,0.1,0.45
"ElasticNet(alpha=0.1, l1_ratio=0.45) (Test)",12.040324,16.276062,0.757481,5.391110e+06,20,0.1,0.45
